# SQLite CRUD Tutorial with Python & Pandas

This notebook demonstrates how to perform **CRUD** (Create, Read, Update, Delete) operations on a SQLite database using Python's `sqlite3` module and `pandas`.

We will work with two tables:
- **`customers`** — stores customer details
- **`orders`** — stores order records linked to customers

Each section opens a fresh database connection, performs its operation, and then **closes the connection** before the next section begins.

---
> **Prerequisites:** `pandas` and `sqlite3` (built into Python's standard library — no installation needed)

## Setup — Imports & Database Path

Run this cell once to import libraries and define the path to the SQLite database file.

In [ ]:
import sqlite3
import pandas as pd

# Path to the SQLite database file.
# SQLite will create this file automatically if it does not exist.
DB_PATH = "./data/store.db"

print(f"Database file: {DB_PATH}")
print(f"pandas version : {pd.__version__}")
print(f"sqlite3 version: {sqlite3.sqlite_version}")

---
## 1. CREATE — Build DataFrames & Write to SQLite Tables

We create two pandas DataFrames — `customers` and `orders` — and persist them as SQLite tables using `DataFrame.to_sql()`.

- `if_exists="replace"` drops and recreates the table if it already exists.
- `index=False` prevents pandas from writing the DataFrame row-index as a column.

The connection is **closed** at the end of this section.

In [ ]:
# ── CREATE ──────────────────────────────────────────────────────────────────
# Build the customers DataFrame
customers_df = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "name":        ["Alice Martin", "Bob Singh", "Carol White", "David Kim", "Eva Brown"],
    "email":       ["alice@example.com", "bob@example.com", "carol@example.com",
                    "david@example.com", "eva@example.com"],
    "city":        ["Halifax", "Toronto", "Vancouver", "Calgary", "Ottawa"]
})

# Build the orders DataFrame
orders_df = pd.DataFrame({
    "order_id":    [101, 102, 103, 104, 105, 106],
    "customer_id": [1,   2,   1,   3,   5,   2  ],
    "product":     ["Laptop", "Mouse", "Monitor", "Keyboard", "Webcam", "Headset"],
    "amount":      [1299.99, 29.99, 349.99, 79.99, 89.99, 59.99],
    "order_date":  ["2024-01-15", "2024-01-20", "2024-02-03",
                    "2024-02-10", "2024-03-05", "2024-03-18"]
})

print("customers_df:")
print(customers_df.to_string(index=False))
print("\norders_df:")
print(orders_df.to_string(index=False))

In [ ]:
# Open connection and write both DataFrames to SQLite tables
conn = sqlite3.connect(DB_PATH)

customers_df.to_sql("customers", conn, if_exists="replace", index=False)
orders_df.to_sql("orders", conn, if_exists="replace", index=False)

print("✅ Tables 'customers' and 'orders' written to the database.")

# Verify the tables exist
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
tables = cursor.fetchall()
print("Tables in database:", [t[0] for t in tables])

# ── CLOSE connection after CREATE ───────────────────────────────────────────
conn.close()
print("🔒 Connection closed.")

---
## 2. READ — Query Tables with pandas

We reopen the connection and use `pd.read_sql()` to execute SELECT queries and load the results directly into DataFrames.

Three examples are shown:
1. Read the full `customers` table
2. Read the full `orders` table
3. A filtered query — orders over $100

The connection is **closed** at the end of this section.

In [ ]:
# ── READ ─────────────────────────────────────────────────────────────────────
# Reopen connection
conn = sqlite3.connect(DB_PATH)
print("🔓 Connection opened.\n")

# 2a. Read all customers
print("── All Customers ──")
df_customers = pd.read_sql("SELECT * FROM customers;", conn)
print(df_customers.to_string(index=False))

# 2b. Read all orders
print("\n── All Orders ──")
df_orders = pd.read_sql("SELECT * FROM orders;", conn)
print(df_orders.to_string(index=False))

# 2c. Filtered query: orders with amount > 100
print("\n── Orders Over $100.00 ──")
df_large = pd.read_sql("SELECT * FROM orders WHERE amount > 100.00;", conn)
print(df_large.to_string(index=False))

# ── CLOSE connection after READ ──────────────────────────────────────────────
conn.close()
print("\n🔒 Connection closed.")

---
## 3. UPDATE — Modify Existing Records

We reopen the connection and use `cursor.execute()` with SQL `UPDATE` statements to modify data already in the database.

Two updates are demonstrated:
1. Change a customer's city
2. Apply a 10% discount to all orders for a specific customer

After each update, we re-read the affected rows to confirm the change.

The connection is **closed** at the end of this section.

In [ ]:
# ── UPDATE ───────────────────────────────────────────────────────────────────
# Reopen connection
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
print("🔓 Connection opened.\n")

# 3a. Update a customer's city (customer_id = 4 moved from Calgary to Edmonton)
print("── Update 1: Change David Kim's city from Calgary → Edmonton ──")
cursor.execute("""
    UPDATE customers
    SET city = 'Edmonton'
    WHERE customer_id = 4;
""")
conn.commit()
print(f"   Rows affected: {cursor.rowcount}")

# Verify the change
df_check = pd.read_sql("SELECT * FROM customers WHERE customer_id = 4;", conn)
print(df_check.to_string(index=False))

# 3b. Apply a 10% discount to all orders belonging to customer_id = 2 (Bob Singh)
print("\n── Update 2: Apply 10% discount to all of Bob Singh's orders ──")
print("   Before:")
df_before = pd.read_sql("SELECT * FROM orders WHERE customer_id = 2;", conn)
print(df_before.to_string(index=False))

cursor.execute("""
    UPDATE orders
    SET amount = ROUND(amount * 0.90, 2)
    WHERE customer_id = 2;
""")
conn.commit()
print(f"\n   Rows affected: {cursor.rowcount}")

print("   After:")
df_after = pd.read_sql("SELECT * FROM orders WHERE customer_id = 2;", conn)
print(df_after.to_string(index=False))

# ── CLOSE connection after UPDATE ────────────────────────────────────────────
conn.close()
print("\n🔒 Connection closed.")

---
## 4. DELETE — Remove Records

We reopen the connection and use SQL `DELETE` statements to remove rows from the database.

Two deletions are shown:
1. Delete a single order by `order_id`
2. Delete a customer (and all of their orders) using a subquery — demonstrating referential integrity clean-up

We verify the remaining records after each deletion.

The connection is **closed** at the end of this section.

In [ ]:
# ── DELETE ───────────────────────────────────────────────────────────────────
# Reopen connection
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
print("🔓 Connection opened.\n")

# 4a. Delete a single order (order_id = 103 — Alice's Monitor order)
print("── Delete 1: Remove order_id = 103 (Monitor) ──")
print("   Orders before deletion:")
df_before = pd.read_sql("SELECT * FROM orders ORDER BY order_id;", conn)
print(df_before.to_string(index=False))

cursor.execute("DELETE FROM orders WHERE order_id = 103;")
conn.commit()
print(f"\n   Rows deleted: {cursor.rowcount}")

print("   Orders after deletion:")
df_after = pd.read_sql("SELECT * FROM orders ORDER BY order_id;", conn)
print(df_after.to_string(index=False))

# 4b. Delete customer_id = 5 (Eva Brown) and her associated orders
print("\n── Delete 2: Remove customer_id = 5 (Eva Brown) and her orders ──")

# First remove her orders (clean up child records before the parent)
cursor.execute("DELETE FROM orders WHERE customer_id = 5;")
conn.commit()
print(f"   Eva's orders deleted: {cursor.rowcount} row(s)")

# Then remove the customer row
cursor.execute("DELETE FROM customers WHERE customer_id = 5;")
conn.commit()
print(f"   Eva's customer record deleted: {cursor.rowcount} row(s)")

print("\n   Remaining customers:")
df_cust = pd.read_sql("SELECT * FROM customers ORDER BY customer_id;", conn)
print(df_cust.to_string(index=False))

print("\n   Remaining orders:")
df_ord = pd.read_sql("SELECT * FROM orders ORDER BY order_id;", conn)
print(df_ord.to_string(index=False))

# ── CLOSE connection after DELETE ────────────────────────────────────────────
conn.close()
print("\n🔒 Connection closed.")

---
## 5. JOIN — Combine Tables & Save Results to a Third Table

We reopen the connection and write a SQL `JOIN` query that combines `customers` and `orders` into a single result set. That result is then saved as a new third table called `customer_orders` using `DataFrame.to_sql()`.

**Query goal:** produce a flat summary showing each order alongside the customer's name and city, with a total spend column grouped per customer.

Two steps:
1. `INNER JOIN` — produces a detailed row-per-order view, saved as `customer_orders`
2. Aggregation query — reads `customer_orders` back and summarises spend per customer

The connection is **closed** at the end of this section.

In [ ]:
# ── JOIN & WRITE TO THIRD TABLE ───────────────────────────────────────────────
# Reopen connection
conn = sqlite3.connect(DB_PATH)
print("🔓 Connection opened.\n")

# 5a. INNER JOIN customers ⟶ orders to produce a flat detail view
join_sql = """
    SELECT
        c.customer_id,
        c.name        AS customer_name,
        c.city,
        o.order_id,
        o.product,
        o.amount,
        o.order_date
    FROM customers AS c
    INNER JOIN orders AS o
        ON c.customer_id = o.customer_id
    ORDER BY c.customer_id, o.order_id;
"""

df_joined = pd.read_sql(join_sql, conn)

print("── Joined Result (customers INNER JOIN orders) ──")
print(df_joined.to_string(index=False))

# 5b. Write the joined result to a new table: customer_orders
df_joined.to_sql("customer_orders", conn, if_exists="replace", index=False)
print(f"\n✅ Joined result written to new table 'customer_orders' ({len(df_joined)} rows).")

# Verify all three tables now exist
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
print("Tables in database:", [t[0] for t in cursor.fetchall()])

# 5c. Read back customer_orders and compute a spend summary per customer
print("\n── Spend Summary per Customer (from customer_orders) ──")
summary_sql = """
    SELECT
        customer_name,
        city,
        COUNT(order_id)        AS total_orders,
        ROUND(SUM(amount), 2)  AS total_spent
    FROM customer_orders
    GROUP BY customer_id
    ORDER BY total_spent DESC;
"""
df_summary = pd.read_sql(summary_sql, conn)
print(df_summary.to_string(index=False))

# ── CLOSE connection after JOIN ───────────────────────────────────────────────
conn.close()
print("\n🔒 Connection closed.")

---
## 6. Cleanup — Remove the Database File (Optional)

Run this cell if you want to delete the `store.db` file and start fresh. Skip it if you want to keep the database for further exploration.

In [ ]:
import os

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(f"🗑️  '{DB_PATH}' deleted. Run the CREATE cell again to start over.")
else:
    print(f"'{DB_PATH}' does not exist — nothing to delete.")